In [1]:
# ============================================================
# AMBIENT TEMPERATURE SYSTEM FAILURE
# NLP + TF-IDF + K-MEANS CLUSTERING
# Complete Jupyter Notebook Code
# ============================================================


# ============================================================
# CELL 1: INSTALL REQUIRED LIBRARIES
# ============================================================

# Run this cell if the libraries are not already installed.

# !pip install pandas numpy matplotlib seaborn scikit-learn nltk


# ============================================================
# CELL 2: IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import string
import warnings

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)
from sklearn.preprocessing import StandardScaler

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


# ============================================================
# CELL 3: DOWNLOAD NLTK RESOURCES
# ============================================================

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("NLTK resources downloaded.")


# ============================================================
# CELL 4: LOAD DATASET
# ============================================================

file_path = "ambient_temperature_system_failure.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print()
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())


# ============================================================
# CELL 5: DATASET INFORMATION
# ============================================================

print("========== DATASET INFORMATION ==========")

print("\nShape:")
print(df.shape)

print("\nColumns:")
for column in df.columns:
    print("-", column)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())


# ============================================================
# CELL 6: STATISTICAL SUMMARY
# ============================================================

print("========== STATISTICAL SUMMARY ==========")

display(df.describe(include="all"))


# ============================================================
# CELL 7: REMOVE DUPLICATES
# ============================================================

before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Duplicate rows removed:", before - after)
print("New dataset shape:", df.shape)


# ============================================================
# CELL 8: IDENTIFY TEXT COLUMNS
# ============================================================

text_columns = df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Text/Object columns found:")
print(text_columns)


# ============================================================
# CELL 9: IDENTIFY NUMERICAL COLUMNS
# ============================================================

numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical columns found:")
print(numeric_columns)


# ============================================================
# CELL 10: AUTOMATICALLY FIND THE TEXT COLUMN
# ============================================================

# Common names for text columns
possible_text_names = [
    "text",
    "message",
    "description",
    "failure_description",
    "failure",
    "comment",
    "comments",
    "event",
    "log",
    "system_failure",
    "error",
    "errors"
]

text_column = None

# First try to find a column with a common text name
for column in df.columns:

    if column.lower() in possible_text_names:
        text_column = column
        break


# If no common name is found, use the first object column
if text_column is None and len(text_columns) > 0:
    text_column = text_columns[0]


print("Selected text column:", text_column)


# ============================================================
# CELL 11: IF THERE IS NO TEXT COLUMN
# ============================================================

if text_column is None:

    print()
    print("WARNING:")
    print("No text column was found in the dataset.")
    print()
    print("Available columns are:")
    print(df.columns.tolist())
    print()
    print("The notebook will create text from numerical/categorical")
    print("information so that the NLP pipeline can still run.")

    # Create a text representation from all available columns
    df["combined_text"] = df.astype(str).agg(" ".join, axis=1)

    text_column = "combined_text"

else:

    print("Using column:", text_column)


# ============================================================
# CELL 12: HANDLE MISSING TEXT VALUES
# ============================================================

df[text_column] = df[text_column].fillna("")

print("Missing text values after cleaning:")
print(df[text_column].isnull().sum())


# ============================================================
# CELL 13: CREATE NLP CLEANING FUNCTION
# ============================================================

stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()


def clean_text(text):

    # Convert to string
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords
    words = [
        word
        for word in words
        if word not in stop_words
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    # Join words
    text = " ".join(words)

    return text


# ============================================================
# CELL 14: APPLY NLP CLEANING
# ============================================================

df["clean_text"] = df[text_column].apply(clean_text)

print("Original text:")
display(df[[text_column]].head())

print()
print("Cleaned text:")
display(df[["clean_text"]].head())


# ============================================================
# CELL 15: REMOVE EMPTY TEXT RECORDS
# ============================================================

before = len(df)

df = df[
    df["clean_text"].str.strip() != ""
].copy()

after = len(df)

print("Empty text rows removed:", before - after)
print("Remaining rows:", after)


# ============================================================
# CELL 16: TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer(
    max_features=2000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

X = vectorizer.fit_transform(
    df["clean_text"]
)

print("TF-IDF completed.")
print()
print("TF-IDF matrix shape:")
print(X.shape)


# ============================================================
# CELL 17: SHOW TF-IDF FEATURES
# ============================================================

feature_names = vectorizer.get_feature_names_out()

print("Number of TF-IDF features:", len(feature_names))

print("\nFirst 50 features:")
print(feature_names[:50])


# ============================================================
# CELL 18: CONVERT TF-IDF TO DATAFRAME
# ============================================================

tfidf_df = pd.DataFrame(
    X.toarray(),
    columns=feature_names
)

print("TF-IDF DataFrame shape:")
print(tfidf_df.shape)

display(tfidf_df.head())


# ============================================================
# CELL 19: ELBOW METHOD
# ============================================================

inertia = []

k_values = range(2, 11)

for k in k_values:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X)

    inertia.append(
        kmeans.inertia_
    )


plt.figure(figsize=(10, 6))

plt.plot(
    k_values,
    inertia,
    marker="o"
)

plt.title(
    "Elbow Method for Selecting K"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Inertia"
)

plt.xticks(
    list(k_values)
)

plt.grid(True)

plt.show()


# ============================================================
# CELL 20: SILHOUETTE SCORE FOR DIFFERENT K
# ============================================================

silhouette_scores = []

for k in range(2, 11):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X)

    score = silhouette_score(
        X,
        labels
    )

    silhouette_scores.append(score)


plt.figure(figsize=(10, 6))

plt.plot(
    range(2, 11),
    silhouette_scores,
    marker="o"
)

plt.title(
    "Silhouette Score for Different K Values"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.xticks(
    range(2, 11)
)

plt.grid(True)

plt.show()


# ============================================================
# CELL 21: PRINT SILHOUETTE SCORES
# ============================================================

for k, score in zip(
    range(2, 11),
    silhouette_scores
):

    print(
        f"K = {k} --> Silhouette Score = {score:.4f}"
    )


# ============================================================
# CELL 22: SELECT BEST K
# ============================================================

best_k = list(
    range(2, 11)
)[np.argmax(silhouette_scores)]

print()
print("Best K according to Silhouette Score:", best_k)


# ============================================================
# CELL 23: APPLY FINAL K-MEANS
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["Cluster"] = kmeans.fit_predict(X)

print("K-Means clustering completed.")
print()
print("Number of clusters:", best_k)


# ============================================================
# CELL 24: CLUSTER COUNTS
# ============================================================

cluster_counts = df["Cluster"].value_counts().sort_index()

print("Records in each cluster:")
print(cluster_counts)


# ============================================================
# CELL 25: CLUSTER DISTRIBUTION GRAPH
# ============================================================

plt.figure(figsize=(10, 6))

sns.countplot(
    data=df,
    x="Cluster"
)

plt.title(
    "Number of Records in Each K-Means Cluster"
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Records"
)

plt.show()


# ============================================================
# CELL 26: CALCULATE CLUSTER QUALITY
# ============================================================

silhouette = silhouette_score(
    X,
    df["Cluster"]
)

davies_bouldin = davies_bouldin_score(
    X.toarray(),
    df["Cluster"]
)

calinski = calinski_harabasz_score(
    X.toarray(),
    df["Cluster"]
)

print("========== CLUSTER QUALITY ==========")

print(
    f"Silhouette Score: {silhouette:.4f}"
)

print(
    f"Davies-Bouldin Index: {davies_bouldin:.4f}"
)

print(
    f"Calinski-Harabasz Score: {calinski:.4f}"
)


# ============================================================
# CELL 27: FIND TOP WORDS FOR EACH CLUSTER
# ============================================================

print("========== TOP WORDS IN EACH CLUSTER ==========")

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for cluster_number in range(best_k):

    top_words = [
        feature_names[index]
        for index in order_centroids[
            cluster_number
       , :20]
    ]

    print()
    print(
        f"Cluster {cluster_number}"
    )

    print(
        ", ".join(top_words)
    )


# ============================================================
# CELL 28: CREATE CLUSTER SUMMARY
# ============================================================

cluster_summary = []

for cluster_number in range(best_k):

    cluster_data = df[
        df["Cluster"] == cluster_number
    ]

    top_indices = order_centroids[
        cluster_number
    ][:10]

    top_words = [
        feature_names[i]
        for i in top_indices
    ]

    cluster_summary.append({
        "Cluster": cluster_number,
        "Number_of_Records": len(cluster_data),
        "Top_Words": ", ".join(top_words)
    })


cluster_summary_df = pd.DataFrame(
    cluster_summary
)

display(cluster_summary_df)


# ============================================================
# CELL 29: PCA DIMENSIONALITY REDUCTION
# ============================================================

pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(
    X.toarray()
)

print("PCA completed.")

print(
    "Explained variance:",
    pca.explained_variance_ratio_
)

print(
    "Total explained variance:",
    pca.explained_variance_ratio_.sum()
)


# ============================================================
# CELL 30: CREATE PCA DATAFRAME
# ============================================================

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Cluster": df["Cluster"].values
})

display(pca_df.head())


# ============================================================
# CELL 31: VISUALIZE K-MEANS CLUSTERS
# ============================================================

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="viridis",
    s=80
)

plt.title(
    "K-Means Clusters using PCA"
)

plt.xlabel(
    "Principal Component 1"
)

plt.ylabel(
    "Principal Component 2"
)

plt.grid(True)

plt.show()


# ============================================================
# CELL 32: DISPLAY RECORDS BY CLUSTER
# ============================================================

for cluster_number in range(best_k):

    print()
    print("=" * 70)
    print(
        f"CLUSTER {cluster_number}"
    )
    print("=" * 70)

    cluster_data = df[
        df["Cluster"] == cluster_number
    ]

    display(
        cluster_data[
            [text_column, "clean_text", "Cluster"]
        ].head(10)
    )


# ============================================================
# CELL 33: TOP TERMS VISUALIZATION
# ============================================================

for cluster_number in range(best_k):

    top_indices = order_centroids[
        cluster_number
    ][:15]

    words = [
        feature_names[i]
        for i in top_indices
    ]

    scores = [
        kmeans.cluster_centers_[
            cluster_number,
            i
        ]
        for i in top_indices
    ]

    plt.figure(figsize=(10, 6))

    plt.barh(
        words[::-1],
        scores[::-1]
    )

    plt.title(
        f"Top 15 Terms - Cluster {cluster_number}"
    )

    plt.xlabel(
        "TF-IDF Importance"
    )

    plt.ylabel(
        "Word"
    )

    plt.show()


# ============================================================
# CELL 34: CLUSTER SUMMARY TABLE
# ============================================================

summary = df.groupby(
    "Cluster"
).size().reset_index(
    name="Number_of_Records"
)

summary["Percentage"] = (
    summary["Number_of_Records"]
    / len(df)
    * 100
)

display(summary)


# ============================================================
# CELL 35: CHECK NUMERICAL FEATURES
# ============================================================

print("Numerical columns available:")

for column in numeric_columns:
    print("-", column)


# ============================================================
# CELL 36: NUMERICAL K-MEANS
# OPTIONAL
# ============================================================

# This section performs K-Means using numerical
# columns separately from the NLP model.

if len(numeric_columns) > 0:

    # Remove any cluster columns that may have been created
    numerical_features = [
        column
        for column in numeric_columns
        if column not in [
            "Cluster",
            "Numeric_Cluster"
        ]
    ]

    if len(numerical_features) > 0:

        numeric_data = df[
            numerical_features
        ].copy()

        # Replace infinite values
        numeric_data = numeric_data.replace(
            [np.inf, -np.inf],
            np.nan
        )

        # Fill missing values with median
        numeric_data = numeric_data.fillna(
            numeric_data.median()
        )

        # Standardization
        scaler = StandardScaler()

        X_numeric = scaler.fit_transform(
            numeric_data
        )

        # K-Means
        numeric_kmeans = KMeans(
            n_clusters=best_k,
            random_state=42,
            n_init=10
        )

        df["Numeric_Cluster"] = (
            numeric_kmeans.fit_predict(
                X_numeric
            )
        )

        print(
            "Numerical K-Means completed."
        )

        print(
            df["Numeric_Cluster"]
            .value_counts()
            .sort_index()
        )


# ============================================================
# CELL 37: COMPARE NLP AND NUMERICAL CLUSTERS
# ============================================================

if "Numeric_Cluster" in df.columns:

    comparison = pd.crosstab(
        df["Cluster"],
        df["Numeric_Cluster"]
    )

    print(
        "NLP Cluster vs Numerical Cluster:"
    )

    display(comparison)


# ============================================================
# CELL 38: HEATMAP OF CLUSTER COMPARISON
# ============================================================

if "Numeric_Cluster" in df.columns:

    plt.figure(figsize=(10, 7))

    sns.heatmap(
        comparison,
        annot=True,
        fmt="d",
        cmap="Blues"
    )

    plt.title(
        "NLP K-Means Cluster vs Numerical K-Means Cluster"
    )

    plt.xlabel(
        "Numerical Cluster"
    )

    plt.ylabel(
        "NLP Cluster"
    )

    plt.show()


# ============================================================
# CELL 39: ADD PCA VALUES TO ORIGINAL DATASET
# ============================================================

df["PCA_1"] = X_pca[:, 0]

df["PCA_2"] = X_pca[:, 1]

print(
    "PCA values added to dataset."
)


# ============================================================
# CELL 40: DISPLAY FINAL DATASET
# ============================================================

print("========== FINAL DATASET ==========")

print(
    "Final shape:",
    df.shape
)

display(df.head(20))


# ============================================================
# CELL 41: SAVE CLUSTERED DATASET
# ============================================================

output_file = (
    "ambient_temperature_system_failure_clustered.csv"
)

df.to_csv(
    output_file,
    index=False
)

print(
    "Dataset saved successfully:"
)

print(
    output_file
)


# ============================================================
# CELL 42: SAVE CLUSTER SUMMARY
# ============================================================

summary_file = (
    "ambient_temperature_cluster_summary.csv"
)

cluster_summary_df.to_csv(
    summary_file,
    index=False
)

print(
    "Cluster summary saved:"
)

print(
    summary_file
)


# ============================================================
# CELL 43: FINAL RESULTS
# ============================================================

print()
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    f"Original dataset rows: {before}"
)

print(
    f"Final dataset rows: {len(df)}"
)

print(
    f"Number of NLP features: {X.shape[1]}"
)

print(
    f"Optimal K: {best_k}"
)

print(
    f"Silhouette Score: {silhouette:.4f}"
)

print(
    f"Davies-Bouldin Index: {davies_bouldin:.4f}"
)

print(
    f"Calinski-Harabasz Score: {calinski:.4f}"
)

print()
print("Cluster distribution:")

print(
    df["Cluster"]
    .value_counts()
    .sort_index()
)

print()
print("Output file:")
print(output_file)

print()
print("Completed successfully!")

Libraries imported successfully.
NLTK resources downloaded.
Dataset loaded successfully.

Number of rows: 7267
Number of columns: 2


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,timestamp,value
0,2013-07-04 00:00:00,69.880835
1,2013-07-04 01:00:00,71.220227
2,2013-07-04 02:00:00,70.877805
3,2013-07-04 03:00:00,68.959400
4,2013-07-04 04:00:00,69.283551


========== DATASET INFORMATION ==========

Shape:
(7267, 2)

Columns:
- timestamp
- value

Data types:
timestamp        str
value        float64
dtype: object

Missing values:
timestamp    0
value        0
dtype: int64

Duplicate rows:
0
========== STATISTICAL SUMMARY ==========


,timestamp,value
count,7267,7267.000000
unique,7267,NaN
top,2013-07-04 00:00:00,NaN
freq,1,NaN
mean,NaN,71.242433
std,NaN,4.247509
min,NaN,57.458406
25%,NaN,68.369411
50%,NaN,71.858493
75%,NaN,74.430958


...

,timestamp
0,2013-07-04 00:00:00
1,2013-07-04 01:00:00
2,2013-07-04 02:00:00
3,2013-07-04 03:00:00
4,2013-07-04 04:00:00



Cleaned text:


,clean_text
0,
1,
2,
3,
4,


Empty text rows removed: 7267
Remaining rows: 0


ValueError: empty vocabulary; perhaps the documents only contain stop words